## E1c - Control Program
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 22, 2026

### About: This is where your trained model(s) become behavior. Same pattern as C1/C2/D2/D3: write a `decide_action` function, drop it into a Start/Stop-button-controlled loop, test.
### The difference today: nothing is pre-filled. Your classes, your flowchart, your decision logic -- entirely up to you and your team.

### Flowchart first -- more important than ever
### This is the biggest, most open-ended flowchart of the week. There's no single "correct" structure the way there was on Tuesday or Thursday -- multiple valid strategies exist. Plan it on paper before you touch this notebook.
### One constraint: **cap your design at two models.** One model is enough for a lot of designs (e.g. a single classifier that directly tells you "single" vs "stack" vs "empty path"). Two models lets you combine two different judgments (e.g. one model for "is anything in front of me," a second for "is it a single or a stack"). The tools below support one or two -- not three or more.

### Which does your flowchart use?
### This notebook has two sections below: **one model** and **two models**. Run the imports cell, then skip to whichever section matches your design -- you don't need both.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# Imports needed for either section below.

import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser

from jetcam_lite import TraitletCamera, bgr8_to_jpeg

from robot_utils import get_rvr, close_if_exists
from jupyter_utils import register_dlink
from inference_utils import load_model_and_metadata, show_inference_grid
from behavior_utils import (
    start_behavior_loop,
    start_two_stage_behavior_loop,
    create_start_stop_buttons,
    stop_behavior_loop,
)

---
<span style="color: orange; font-size: 55px; font-style: italic;">SECTION A: One-Model Program</span>

### Only run this section if your flowchart uses a single model. If you're using two, skip to Section B below.

### STEP A1. Choose your model.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

model_chooser = FileChooser('/home/explorer/Models/')
model_chooser.filter_pattern = '*.pth'
display(model_chooser)

### STEP A2. Load it, and run a quick accuracy check.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

model, device, class_names, training_record = load_model_and_metadata(model_chooser.selected)
print(f"Classes: {class_names}")

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

show_inference_grid(model, class_names, device, training_record['dataset_dir'])

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP A3. Connect to the robot and start the camera.

rvr = get_rvr()
rvr.reset_yaw()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

### STEP A4. Write your decision logic. `decide_action` is called repeatedly (about twice a second) with the model's current prediction as `label`. Your classes are printed above -- use those exact strings.
### Reminder: the RVR needs a new drive command at least every 2 seconds to keep moving. Since this function is called on a fixed cycle, any branch that should keep driving just needs to issue a drive command each time it runs -- the repetition handles the rest.

In [ ]:
#### ------> ACTIVITY E1c.1: decide_action function (one model) <-----#####
# About: called repeatedly while the behavior is running. label is
# whatever your model just predicted -- one of the class_names printed
# above.

##### INSTRUCTIONS: #####
# Write the full decision logic for your flowchart. A starting pattern:
#
#   if label == 'your_class_name':
#       rvr.raw_motors(...)            # or rvr.drive_with_heading(...)
#   else:
#       ...

#<<<<<< replace this with your code >>>>>>
def decide_action(rvr, label):
    pass

### STEP A5. Build the Start/Stop buttons. Running this cell does **not** move the robot -- nothing happens until you press Start.

######## WILL CAUSE ROBOT MOTION ONCE STARTED: ENSURE ROBOT IS ON THE GROUND WITH ROOM TO MOVE #########

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

buttons = create_start_stop_buttons(
    lambda: start_behavior_loop(rvr, camera, model, class_names, device, decide_action)
)
display(buttons)

### Press **Start** to test, **Stop** to immediately halt. When you're done with this section, run the cell below.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
stop_behavior_loop()

---
<span style="color: orange; font-size: 55px; font-style: italic;">SECTION B: Two-Model Program</span>

### Only run this section if your flowchart uses two models -- a primary model, and a secondary model that only runs when the primary hits some trigger label (e.g. primary says "blocked," so you check a secondary model to decide what kind of obstacle it is).

### STEP B1. Choose both models.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

print("Choose your PRIMARY model:")
primary_chooser = FileChooser('/home/explorer/Models/')
primary_chooser.filter_pattern = '*.pth'
display(primary_chooser)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

print("Choose your SECONDARY model:")
secondary_chooser = FileChooser('/home/explorer/Models/')
secondary_chooser.filter_pattern = '*.pth'
display(secondary_chooser)

### STEP B2. Load both, and run a quick accuracy check on each.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

primary_model, primary_device, primary_classes, primary_record = \
    load_model_and_metadata(primary_chooser.selected)
print(f"Primary classes: {primary_classes}")

print()

secondary_model, secondary_device, secondary_classes, secondary_record = \
    load_model_and_metadata(secondary_chooser.selected)
print(f"Secondary classes: {secondary_classes}")

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

show_inference_grid(primary_model, primary_classes, primary_device, primary_record['dataset_dir'])

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

show_inference_grid(secondary_model, secondary_classes, secondary_device, secondary_record['dataset_dir'])

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP B3. Connect to the robot and start the camera.

rvr = get_rvr()
rvr.reset_yaw()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

### STEP B4. Write your decision logic. `decide_action` receives `primary_label` (always present) and `secondary_label` (only set when `primary_label` matches your chosen `trigger_label` below -- otherwise `None`).
### If your design needs to track a heading across turns (like D2/D3 did), `heading_state = {'current': 0}` is a useful pattern -- update it every time you turn, and read it every time you drive forward, so "keep going" always means "continue in whatever direction I'm currently facing."

In [ ]:
#### ------> ACTIVITY E1c.2: decide_action function (two models) <-----#####
# About: called repeatedly while the behavior is running. primary_label
# is your primary model's current prediction (always present).
# secondary_label is your secondary model's prediction -- only present
# (not None) when primary_label == trigger_label (set below).

##### INSTRUCTIONS: #####
# Write the full decision logic for your flowchart. A starting pattern:
#
#   if primary_label == 'your_trigger_label':
#       if secondary_label == 'something':
#           ...
#   else:
#       ...

heading_state = {'current': 0}

#<<<<<< replace this with your code >>>>>>
def decide_action(rvr, primary_label, secondary_label):
    pass

### Set `trigger_label` to whichever primary-model class should trigger the secondary model.

In [ ]:
##### ----- FEEL FREE TO CHANGE THIS VALUE ----- #####
trigger_label = "blocked"

### STEP B5. Build the Start/Stop buttons. Running this cell does **not** move the robot -- nothing happens until you press Start.

######## WILL CAUSE ROBOT MOTION ONCE STARTED: ENSURE ROBOT IS ON THE GROUND WITH ROOM TO MOVE #########

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

buttons = create_start_stop_buttons(
    lambda: start_two_stage_behavior_loop(
        rvr, camera,
        primary_model, primary_classes, primary_device,
        secondary_model, secondary_classes, secondary_device,
        decide_action,
        trigger_label=trigger_label
    )
)
display(buttons)

### Press **Start** to test, **Stop** to immediately halt.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
stop_behavior_loop()

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>

### Talk through these questions with your team:
### - Did your flowchart survive contact with the real arena? What's different between planning it on paper and watching it run?
### - If something didn't work, is it a model problem (misclassification), a logic problem (the flowchart itself), or a physical problem (speed, timing, distance)? How can you tell the difference?
### - If you have time to iterate: what's the single highest-value change you'd make first?

## This is it -- your robot is running a program you designed, trained, and built entirely yourself this week.

### Wrapping Up
When you're done, run the cell below to release the robot's connection.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed.")